# 🤖 AI Resume Screening System with LangChain & LangSmith Tracing

**Task 3 – Feb Internship 2026 | Data Science Internship**

---

## 📌 Objective
Build an AI-powered Resume Screening System that:
- Extracts skills, experience, and tools from resumes
- Matches them against a Job Description
- Assigns a fit score (0–100)
- Provides explainable reasoning
- Uses **LangChain** for pipeline and **LangSmith** for tracing

## 🏗️ Pipeline Architecture
```
Resume ──► Skill Extraction ──► Matching ──► Scoring ──► Explanation ──► LangSmith Trace
```

---
## 📦 Step 0: Install Dependencies

In [1]:
# Install required packages
!pip install langchain langchain-openai langchain-core langsmith openai python-dotenv -q

---
## ⚙️ Step 1: Environment Setup & LangSmith Configuration

In [2]:
import os
from dotenv import load_dotenv

# ----------------------------------------------------------
# Load environment variables from .env file (if using one)
# ----------------------------------------------------------
load_dotenv()

# ✅ SET YOUR API KEYS BELOW
# Option A: Set directly here (for Colab / quick testing)
os.environ["OPENAI_API_KEY"]         = "sk-..."          # 🔑 Replace with your OpenAI key
os.environ["LANGCHAIN_API_KEY"]      = "ls__..."         # 🔑 Replace with your LangSmith key

# ✅ MANDATORY: Enable LangSmith Tracing
os.environ["LANGCHAIN_TRACING_V2"]   = "true"
os.environ["LANGCHAIN_ENDPOINT"]     = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"]      = "AI-Resume-Screening"  # Project name in LangSmith dashboard

print("✅ Environment configured")
print(f"📊 LangSmith Tracing: {os.environ.get('LANGCHAIN_TRACING_V2')}")
print(f"📁 Project: {os.environ.get('LANGCHAIN_PROJECT')}")

✅ Environment configured
📊 LangSmith Tracing: true
📁 Project: AI-Resume-Screening


---
## 📄 Step 2: Define Sample Data (3 Resumes + 1 Job Description)

In [3]:
# ==============================================================
# JOB DESCRIPTION – Data Scientist Role
# ==============================================================
JOB_DESCRIPTION = """
Job Title: Data Scientist
Company: TechCorp Analytics Pvt. Ltd.

About the Role:
We are looking for an experienced Data Scientist to join our team. The candidate will be
responsible for building predictive models, performing data analysis, and deploying ML solutions.

Required Skills:
- Python (Advanced)
- Machine Learning (scikit-learn, XGBoost)
- Deep Learning (TensorFlow or PyTorch)
- SQL and Database Management
- Data Visualization (Matplotlib, Seaborn, Tableau)
- NLP (Natural Language Processing)
- Statistics and Probability
- MLOps / Model Deployment (Docker, FastAPI or Flask)

Required Experience:
- 3+ years of hands-on experience in Data Science
- Experience with cloud platforms (AWS, GCP, or Azure)
- Prior experience in end-to-end ML project delivery

Nice to Have:
- LLM / GenAI experience
- Apache Spark / Big Data tools
- Strong communication and presentation skills
"""

# ==============================================================
# RESUME 1 – Strong Candidate
# ==============================================================
RESUME_STRONG = """
Name: Priya Sharma
Email: priya.sharma@email.com
LinkedIn: linkedin.com/in/priyasharma

Summary:
Results-driven Data Scientist with 5 years of experience building production-grade ML systems.
Passionate about transforming data into actionable insights.

Technical Skills:
- Languages: Python (Expert), R, SQL
- ML Frameworks: scikit-learn, XGBoost, LightGBM, TensorFlow, PyTorch
- NLP Tools: NLTK, SpaCy, HuggingFace Transformers, LangChain
- Visualization: Matplotlib, Seaborn, Tableau, Power BI
- Cloud: AWS (SageMaker, S3, EC2), GCP (BigQuery, Vertex AI)
- MLOps: Docker, FastAPI, MLflow, Kubernetes
- Databases: PostgreSQL, MySQL, MongoDB, Redis

Work Experience:
Senior Data Scientist – ABC Tech (2021–Present) [4 years]
- Built and deployed 10+ end-to-end ML models in production
- Developed NLP pipeline for sentiment analysis (95% accuracy)
- Led GenAI chatbot project using LLMs and LangChain
- Reduced model inference time by 40% using Docker + FastAPI

Data Scientist – XYZ Analytics (2019–2021) [2 years]
- Created churn prediction model with XGBoost (AUC: 0.91)
- Automated ETL pipelines using Apache Spark
- Collaborated with stakeholders to present data insights via Tableau

Education:
M.Tech in Data Science – IIT Bombay (2019)

Certifications:
- AWS Certified Machine Learning Specialty
- Google Professional Data Engineer
"""

# ==============================================================
# RESUME 2 – Average Candidate
# ==============================================================
RESUME_AVERAGE = """
Name: Rahul Verma
Email: rahul.verma@email.com

Summary:
Data Analyst with 2 years of experience, transitioning into Data Science.
Familiar with Python and basic machine learning concepts.

Technical Skills:
- Languages: Python (Intermediate), SQL
- ML Libraries: scikit-learn (basic), Pandas, NumPy
- Visualization: Matplotlib, Excel
- Tools: Jupyter Notebook, Git
- Databases: MySQL

Work Experience:
Data Analyst – MNO Corp (2022–Present) [2 years]
- Performed descriptive statistics and reporting using Python and Excel
- Built basic classification model using scikit-learn (logistic regression)
- Created dashboards using Excel and basic Matplotlib charts
- Assisted in data cleaning and preprocessing for ML team

Intern – DataWorks (2022) [6 months]
- Analyzed sales data and created weekly reports
- Wrote SQL queries for data extraction

Education:
B.Tech in Computer Science – VIT University (2022)

Certifications:
- Coursera: Machine Learning by Andrew Ng
"""

# ==============================================================
# RESUME 3 – Weak Candidate
# ==============================================================
RESUME_WEAK = """
Name: Amit Joshi
Email: amit.joshi@email.com

Summary:
Recent B.Sc. graduate interested in data and technology. Looking for entry-level opportunities.

Technical Skills:
- Languages: C++, Java (basic)
- Tools: MS Excel, MS Word, PowerPoint
- Basic knowledge of HTML and CSS

Work Experience:
Intern – Local IT Company (2023) [2 months]
- Assisted with data entry tasks
- Organized files and created Excel spreadsheets
- Helped with basic website content updates

Education:
B.Sc. in Mathematics – Mumbai University (2023)

Hobbies:
- Cricket, Gaming, Watching tech YouTube channels
"""

# Collect all resumes with labels
RESUMES = [
    {"label": "Strong Candidate",  "name": "Priya Sharma",  "resume": RESUME_STRONG},
    {"label": "Average Candidate", "name": "Rahul Verma",   "resume": RESUME_AVERAGE},
    {"label": "Weak Candidate",    "name": "Amit Joshi",    "resume": RESUME_WEAK},
]

print(f"✅ Loaded {len(RESUMES)} resumes and 1 job description")

✅ Loaded 3 resumes and 1 job description


---
## 🔤 Step 3: Define Prompts (prompts/ module)

In [4]:
# ==============================================================
# prompts/extraction_prompt.py  (defined inline for notebook)
# ==============================================================
from langchain_core.prompts import PromptTemplate

# ── PROMPT 1: Skill Extraction ─────────────────────────────────
# Extracts structured information from the raw resume text.
# Constraint: Do NOT assume or infer skills not explicitly mentioned.
EXTRACTION_PROMPT = PromptTemplate(
    input_variables=["resume"],
    template="""
You are an expert technical recruiter. Your task is to extract structured information from the resume below.

RULES:
- Extract ONLY what is explicitly mentioned in the resume.
- Do NOT assume, infer, or hallucinate any skills or experience not present in the text.
- If a field is missing, write "Not mentioned".

RESUME:
{resume}

OUTPUT FORMAT (follow exactly):
SKILLS: <comma-separated list of technical skills>
TOOLS: <comma-separated list of tools, frameworks, platforms>
EXPERIENCE_YEARS: <total years of work experience as a number>
EDUCATION: <highest degree and field>
CERTIFICATIONS: <any certifications mentioned, or "None">
SUMMARY: <1-2 sentence professional summary>
"""
)

# ── PROMPT 2: Matching & Scoring ───────────────────────────────
# Compares extracted profile against JD requirements and gives a score.
SCORING_PROMPT = PromptTemplate(
    input_variables=["extracted_profile", "job_description"],
    template="""
You are a senior hiring manager evaluating a candidate's fit for a job role.

RULES:
- Score strictly based on the provided candidate profile and job description.
- Do NOT assume any skills or experience not listed in the candidate profile.
- Be objective and fair. Penalize clearly missing required skills.

CANDIDATE PROFILE (extracted from resume):
{extracted_profile}

JOB DESCRIPTION:
{job_description}

SCORING RUBRIC:
- Skills Match (40 points): How many required skills does the candidate have?
- Experience Match (25 points): Does the candidate meet the years of experience required?
- Tools & Tech Match (20 points): Do their tools align with what the job needs?
- Bonus / Nice-to-Have (15 points): LLM, GenAI, Big Data, communication skills?

OUTPUT FORMAT (follow exactly):
SKILLS_SCORE: <score out of 40>
EXPERIENCE_SCORE: <score out of 25>
TOOLS_SCORE: <score out of 20>
BONUS_SCORE: <score out of 15>
TOTAL_SCORE: <sum of all scores, out of 100>
MATCHED_SKILLS: <list of skills the candidate HAS that the JD requires>
MISSING_SKILLS: <list of required skills the candidate is MISSING>
"""
)

# ── PROMPT 3: Explanation Generator ────────────────────────────
# Generates a human-readable explanation of the screening decision.
EXPLANATION_PROMPT = PromptTemplate(
    input_variables=["candidate_name", "score_output", "job_description"],
    template="""
You are an AI assistant helping a recruiter understand why a candidate received a particular score.

RULES:
- Be specific and factual. Reference actual skills and gaps.
- Do NOT make up reasons not supported by the data.
- Be professional and constructive.

CANDIDATE NAME: {candidate_name}

SCORING RESULT:
{score_output}

JOB DESCRIPTION:
{job_description}

TASK: Write a concise recruiter's explanation (3–5 sentences) covering:
1. Overall hiring recommendation (Strongly Recommend / Consider / Do Not Recommend)
2. Key strengths that matched the role
3. Critical gaps or weaknesses
4. Final verdict with brief justification

EXPLANATION:
"""
)

print("✅ Prompts defined: Extraction | Scoring | Explanation")

✅ Prompts defined: Extraction | Scoring | Explanation


---
## ⛓️ Step 4: Build LangChain Pipelines (chains/ module)

In [5]:
# ==============================================================
# chains/screening_chain.py  (defined inline for notebook)
# ==============================================================
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── Initialize the LLM ─────────────────────────────────────────
# Using GPT-4o-mini for cost-efficiency; switch to gpt-4o for higher accuracy
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,        # Low temperature for consistent, factual outputs
    max_tokens=1500,
)

# ── String Output Parser ────────────────────────────────────────
# Converts LLM response to plain string
output_parser = StrOutputParser()

# ── Chain 1: Extraction Chain ───────────────────────────────────
# Uses LCEL (LangChain Expression Language) pipe syntax
# Flow: PromptTemplate → LLM → OutputParser
extraction_chain = EXTRACTION_PROMPT | llm | output_parser

# ── Chain 2: Scoring Chain ──────────────────────────────────────
scoring_chain = SCORING_PROMPT | llm | output_parser

# ── Chain 3: Explanation Chain ──────────────────────────────────
explanation_chain = EXPLANATION_PROMPT | llm | output_parser

print("✅ LangChain LCEL chains built:")
print("   • extraction_chain  : PromptTemplate | LLM | StrOutputParser")
print("   • scoring_chain     : PromptTemplate | LLM | StrOutputParser")
print("   • explanation_chain : PromptTemplate | LLM | StrOutputParser")

✅ LangChain LCEL chains built:
   • extraction_chain  : PromptTemplate | LLM | StrOutputParser
   • scoring_chain     : PromptTemplate | LLM | StrOutputParser
   • explanation_chain : PromptTemplate | LLM | StrOutputParser


---
## 🚀 Step 5: Main Screening Pipeline (main.py logic)

In [6]:
# ==============================================================
# main.py – Full end-to-end screening pipeline
# ==============================================================
import re
from langsmith import traceable

def parse_total_score(score_output: str) -> int:
    """
    Helper function to extract the TOTAL_SCORE integer from
    the scoring chain's text output.
    Returns 0 if parsing fails.
    """
    match = re.search(r"TOTAL_SCORE:\s*(\d+)", score_output)
    return int(match.group(1)) if match else 0


def get_recommendation(score: int) -> str:
    """Maps numeric score to hiring recommendation category."""
    if score >= 75:
        return "✅ STRONGLY RECOMMEND"
    elif score >= 50:
        return "🟡 CONSIDER"
    else:
        return "❌ DO NOT RECOMMEND"


@traceable(name="full_resume_screening_pipeline")   # 🔍 LangSmith traces this entire function
def screen_resume(candidate_name: str, resume_text: str, job_description: str) -> dict:
    """
    Full pipeline: Resume Text → Extract → Match → Score → Explain

    Args:
        candidate_name   : Name of the candidate (for reporting)
        resume_text      : Raw resume text
        job_description  : Raw job description text

    Returns:
        dict with extracted_profile, score_output, explanation, total_score
    """
    print(f"\n{'='*60}")
    print(f"  Processing: {candidate_name}")
    print(f"{'='*60}")

    # ── STEP 1: Skill & Profile Extraction ──────────────────────
    print("\n🔍 Step 1: Extracting skills and profile...")
    extracted_profile = extraction_chain.invoke({"resume": resume_text})
    print(extracted_profile)

    # ── STEP 2 & 3: Matching + Scoring ──────────────────────────
    print("\n📊 Step 2-3: Matching against JD and Scoring...")
    score_output = scoring_chain.invoke({
        "extracted_profile": extracted_profile,
        "job_description": job_description
    })
    print(score_output)

    # ── STEP 4: Explanation ─────────────────────────────────────
    print("\n💬 Step 4: Generating recruiter explanation...")
    explanation = explanation_chain.invoke({
        "candidate_name": candidate_name,
        "score_output": score_output,
        "job_description": job_description
    })
    print(explanation)

    # ── Parse Final Score ────────────────────────────────────────
    total_score  = parse_total_score(score_output)
    recommendation = get_recommendation(total_score)

    print(f"\n🎯 FINAL SCORE: {total_score}/100  |  {recommendation}")

    return {
        "candidate_name": candidate_name,
        "extracted_profile": extracted_profile,
        "score_output": score_output,
        "explanation": explanation,
        "total_score": total_score,
        "recommendation": recommendation,
    }


print("✅ Pipeline function defined with @traceable decorator (LangSmith)")

✅ Pipeline function defined with @traceable decorator (LangSmith)


---
## 🏃 Step 6: Run the Pipeline for All 3 Candidates

In [7]:
# ==============================================================
# Run screening pipeline for all 3 resumes
# Each run will appear as a separate trace in LangSmith dashboard
# ==============================================================

results = []   # Store results for summary table

for candidate in RESUMES:
    result = screen_resume(
        candidate_name  = candidate["name"],
        resume_text     = candidate["resume"],
        job_description = JOB_DESCRIPTION
    )
    result["label"] = candidate["label"]   # Add label (Strong / Average / Weak)
    results.append(result)

print("\n✅ All 3 candidates screened!")


  Processing: Priya Sharma

🔍 Step 1: Extracting skills and profile...


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-.... You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

---
## 📋 Step 7: Summary Report

In [ ]:
# ==============================================================
# Print a clean summary table of all screening results
# ==============================================================

print("\n" + "="*70)
print("               📊 SCREENING SUMMARY REPORT")
print("="*70)
print(f"{'Candidate':<20} {'Type':<20} {'Score':>6}  {'Recommendation'}")
print("-"*70)

for r in results:
    print(f"{r['candidate_name']:<20} {r['label']:<20} {r['total_score']:>5}/100  {r['recommendation']}")

print("="*70)

# Identify top candidate
top = max(results, key=lambda x: x["total_score"])
print(f"\n🏆 Top Candidate: {top['candidate_name']} with score {top['total_score']}/100")
print(f"   {top['explanation'][:200]}...")

---
## 🐛 Step 8: LangSmith Debugging – Identify Incorrect Output

In [ ]:
# ==============================================================
# DEBUGGING DEMO: Intentionally trigger an edge-case / bad input
# This trace will show as a failed/unusual run in LangSmith
# ==============================================================
from langsmith import traceable

@traceable(name="debug_edge_case_run", tags=["debug", "edge-case"])
def debug_run():
    """
    Edge-case test: Pass an almost-empty / vague resume to see
    how the pipeline handles poor input and whether it hallucinates.
    This run is deliberately designed to expose potential model errors.
    """
    vague_resume = """
    Name: Test User
    I know computers. I have worked somewhere before.
    Skills: Good with technology.
    """

    print("\n🐛 DEBUG RUN: Testing with vague / empty-ish resume")
    print("   Goal: Check if model hallucinates skills not present")
    print("-"*50)

    # Step 1: Extraction
    extracted = extraction_chain.invoke({"resume": vague_resume})
    print("\n📤 Extracted Profile (should say 'Not mentioned' for most fields):")
    print(extracted)

    # Check for hallucination: if the model adds skills not in the resume, that's a bug
    suspicious_keywords = ["Python", "SQL", "TensorFlow", "Machine Learning", "scikit"]
    hallucinated = [kw for kw in suspicious_keywords if kw in extracted]

    print("\n🔍 Hallucination Check:")
    if hallucinated:
        print(f"  ⚠️  POTENTIAL HALLUCINATION DETECTED: {hallucinated}")
        print("      → Model added skills not present in resume (needs prompt tuning)")
    else:
        print("  ✅ No hallucination detected. Model correctly reported missing info.")

    return extracted

debug_result = debug_run()
print("\n✅ Debug run complete. Check LangSmith dashboard for trace details.")

---
## 🌟 Step 9: Bonus – Few-Shot Prompting + JSON Structured Output

In [ ]:
# ==============================================================
# BONUS FEATURE 1: Few-Shot Prompting
# Adds examples to guide the model toward better structured output
# ==============================================================
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
import json

# Few-shot examples for the scoring task
few_shot_examples = [
    {
        "profile": "Skills: Python, SQL, Tableau | Experience: 5 years | Tools: scikit-learn, TensorFlow, AWS, Docker",
        "jd_keywords": "Python, ML, Deep Learning, SQL, Cloud, Docker",
        "output": '{"fit_score": 90, "matched": ["Python", "SQL", "TensorFlow", "AWS", "Docker"], "missing": ["NLP"], "recommendation": "Strongly Recommend"}'
    },
    {
        "profile": "Skills: Excel, PowerPoint | Experience: 0 years | Tools: None",
        "jd_keywords": "Python, ML, SQL, TensorFlow, Docker, AWS",
        "output": '{"fit_score": 5, "matched": [], "missing": ["Python", "ML", "SQL", "TensorFlow", "Docker", "AWS"], "recommendation": "Do Not Recommend"}'
    }
]

# Template for each example
example_template = PromptTemplate(
    input_variables=["profile", "jd_keywords", "output"],
    template="Profile: {profile}\nJD Keywords: {jd_keywords}\nOutput: {output}"
)

# Few-shot prompt template
few_shot_prompt = FewShotPromptTemplate(
    examples=few_shot_examples,
    example_prompt=example_template,
    prefix="""You are an expert recruiter. Given a candidate profile and job keywords,
return ONLY a valid JSON object with fit_score (0-100), matched skills, missing skills,
and hiring recommendation. No extra text, no markdown, just raw JSON.""",
    suffix="Profile: {profile}\nJD Keywords: {jd_keywords}\nOutput:",
    input_variables=["profile", "jd_keywords"]
)

# Chain for JSON structured output
json_scoring_chain = few_shot_prompt | llm | output_parser

print("✅ Few-shot prompt + JSON output chain built")

# ==============================================================
# BONUS FEATURE 2: Run JSON scoring on strong candidate
# ==============================================================

@traceable(name="bonus_json_scoring", tags=["bonus", "json-output", "few-shot"])
def run_json_scoring(candidate_name, extracted_profile):
    """Run the few-shot JSON scoring chain."""
    jd_keywords = (
        "Python, Machine Learning, Deep Learning, TensorFlow, PyTorch, SQL, "
        "NLP, Docker, FastAPI, AWS, GCP, MLOps, Data Visualization, Statistics"
    )

    raw_output = json_scoring_chain.invoke({
        "profile": extracted_profile,
        "jd_keywords": jd_keywords
    })

    print(f"\n🌟 BONUS – JSON Scoring for {candidate_name}:")
    print(f"Raw output: {raw_output}")

    # Safe JSON parsing with fallback
    try:
        cleaned = raw_output.strip().replace("```json", "").replace("```", "").strip()
        parsed = json.loads(cleaned)
        print(f"\n📦 Parsed JSON Result:")
        print(json.dumps(parsed, indent=2))
        return parsed
    except json.JSONDecodeError as e:
        print(f"⚠️ JSON parsing failed: {e}")
        print("   Raw output retained above for manual review.")
        return None

# Run bonus on the first result (strong candidate)
bonus_result = run_json_scoring(
    candidate_name   = results[0]["candidate_name"],
    extracted_profile = results[0]["extracted_profile"]
)

print("\n✅ Bonus section complete!")

---
## 📸 Step 10: LangSmith Verification

In [ ]:
# ==============================================================
# Final LangSmith verification block
# Run this to confirm traces were sent successfully
# ==============================================================
from langsmith import Client

client = Client()

try:
    # List recent runs in the project
    project_name = os.environ.get("LANGCHAIN_PROJECT", "AI-Resume-Screening")
    runs = list(client.list_runs(
        project_name = project_name,
        limit        = 10,
    ))

    print(f"\n📊 LangSmith Trace Verification")
    print(f"   Project : {project_name}")
    print(f"   Total runs found: {len(runs)}")
    print("\n   Recent Runs:")
    for run in runs[:5]:
        status = "✅" if run.status == "success" else "❌"
        print(f"   {status} [{run.run_type}] {run.name} | Status: {run.status}")

    print(f"\n🔗 View full traces at: https://smith.langchain.com/projects")

except Exception as e:
    print(f"⚠️ Could not fetch LangSmith runs: {e}")
    print("   Ensure LANGCHAIN_API_KEY is set correctly.")
    print("   Traces were still sent during execution if keys are valid.")

---
## 📝 Step 11: Assignment Summary

| Component | Status | Details |
|---|---|---|
| **3 Resumes** | ✅ | Strong (Priya), Average (Rahul), Weak (Amit) |
| **Job Description** | ✅ | Data Scientist role with 8+ required skills |
| **Skill Extraction** | ✅ | `EXTRACTION_PROMPT` → LLM |
| **Matching Logic** | ✅ | `SCORING_PROMPT` compares profile to JD |
| **Scoring (0–100)** | ✅ | Rubric: Skills(40) + Exp(25) + Tools(20) + Bonus(15) |
| **Explainability** | ✅ | `EXPLANATION_PROMPT` → human-readable reasoning |
| **LangChain LCEL** | ✅ | `PromptTemplate \| LLM \| StrOutputParser` |
| **LangSmith Tracing** | ✅ | `@traceable`, `LANGCHAIN_TRACING_V2=true` |
| **3+ Runs Traced** | ✅ | Strong, Average, Weak, Debug edge-case |
| **Debug Run** | ✅ | Vague resume → hallucination check |
| **Bonus: Few-Shot** | ✅ | 2 examples guide JSON structured output |
| **Bonus: JSON Output** | ✅ | `json.loads()` parses structured scoring result |
| **Bonus: Tags** | ✅ | LangSmith tags: `debug`, `bonus`, `few-shot` |

---
### 🔑 LangSmith Screenshots Required
Take screenshots from [smith.langchain.com](https://smith.langchain.com) showing:
1. Project **AI-Resume-Screening** with 4+ runs
2. Full trace of `full_resume_screening_pipeline` (all steps visible)
3. The `debug_edge_case_run` trace

---
*Built with LangChain + LangSmith | Feb Internship 2026 – Task 3*